In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, brier_score_loss
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/telco_clean.csv")

# leakage and non-predictive columns (churn_score/label/reason describe the outcome)
drop_cols = ["customerid", "count", "country", "state", "city", "zip_code",
             "lat_long", "latitude", "longitude",
             "churn_label", "churn_score", "churn_reason", "cltv"]

y = df["churn_value"]
X = df.drop(columns=drop_cols + ["churn_value"])
X = pd.get_dummies(X, drop_first=True).astype(float)
print(X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(y_train.mean(), y_test.mean())

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                    eval_metric="logloss", random_state=42)
xgb.fit(X_train, y_train)

In [ ]:
for name, model in [("LogReg", logreg), ("XGBoost", xgb)]:
    proba = model.predict_proba(X_test)[:, 1]
    print(name,
          "ROC-AUC:", round(roc_auc_score(y_test, proba), 3),
          "PR-AUC:", round(average_precision_score(y_test, proba), 3))
    print(classification_report(y_test, model.predict(X_test)))

In [ ]:
xgb_cal = CalibratedClassifierCV(
    XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                  eval_metric="logloss", random_state=42),
    method="isotonic", cv=5)
xgb_cal.fit(X_train, y_train)

raw = xgb.predict_proba(X_test)[:, 1]
cal = xgb_cal.predict_proba(X_test)[:, 1]
print("Brier raw:", round(brier_score_loss(y_test, raw), 4),
      "| calibrated:", round(brier_score_loss(y_test, cal), 4))
print("ROC-AUC calibrated:", round(roc_auc_score(y_test, cal), 3))

In [ ]:
for name, p in [("raw", raw), ("calibrated", cal)]:
    frac, mean = calibration_curve(y_test, p, n_bins=10)
    plt.plot(mean, frac, marker="o", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
plt.xlabel("Predicted probability")
plt.ylabel("Actual churn rate")
plt.legend()
plt.savefig("../reports/figures/calibration_curve.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, show=False)
plt.savefig("../reports/figures/shap_summary.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(xgb_cal, "../models/xgb_calibrated.joblib")
joblib.dump(list(X.columns), "../models/feature_columns.joblib")